# Selection Tool Functions 

## imports

In [87]:
import pandas as pd
import json
import yaml
import math
import jsonschema
from jsonschema import validate
from jsonschema import Draft7Validator, ValidationError
from datetime import date, datetime
from typing import Dict, List, Union, Any, Tuple
from textwrap import indent

## Constants

In [88]:
# Folders
DATA_FOLDER_NAME = 'data'
RULE_FOLDER_NAME = 'rules'
SCHEMA_FOLDER_NAME = 'schemas'
# Files
DATA_FILE_NAME = 'data_2026-01-07.csv'
RULE_FILE_NAME = 'Alpine_Skiing_qualification_rules.yaml'
DATA_SCHEMA_FILE_NAME = 'dataschema.json'
RULE_SCHEMA_FILE_NAME = 'ruleschema.json'
# Paths
DATA_PATH = f'../{DATA_FOLDER_NAME}/{DATA_FILE_NAME}'
RULE_PATH = f'../{RULE_FOLDER_NAME}/{RULE_FILE_NAME}'
DATA_SCHEMA_PATH = f'../{SCHEMA_FOLDER_NAME}/{DATA_SCHEMA_FILE_NAME}'
RULE_SCHEMA_PATH = f'../{SCHEMA_FOLDER_NAME}/{RULE_SCHEMA_FILE_NAME}'

## read functions 

In [89]:
# Function to read data from CSV
from altair import value


def load_data(data_path):
    data = pd.read_csv(data_path)
    data = data.where(pd.notnull(data), None)
    return data

# Function to read JSON schema
def load_json_schema(schema_path):
    with open(schema_path, 'r') as f:
        return json.load(f)

# Function to read YAML rules
def load_yaml_rules(rules_path):
    with open(rules_path, 'r') as f:
        return yaml.safe_load(f)

# A function that takes data and the schema and removes all invalid columns in the data frame based on the schema
def clean_data(data, schema):
    valid_columns = [prop for prop in schema['properties'].keys() if prop in data.columns]
    # remove all rows with Team Members == Yes
    cleaned_data = data[valid_columns].copy()
    cleaned_data = cleaned_data[cleaned_data["Team Members"] != "Yes"]
    # remove column "Team Members"
    cleaned_data = cleaned_data.drop(columns=["Team Members"])
    # in rank column, convert all values to integers, DNS, DNF, DSQ to be converted to 997, 998, 999 respectively
    def convert_rank(value):
        if value in ['DNS', 'dns']:
            return 996
        elif value in ['DNF', 'dnf']:
            return 997
        elif value in ['DSQ', 'dsq']:
            return 998
        elif value is None:
            return 999
        # Handle actual NaN from pandas
        if isinstance(value, float) and math.isnan(value):
            return 999
        try:
            return int(value)
        except:
            return 999

    cleaned_data['Rank'] = cleaned_data['Rank'].apply(convert_rank)

    def clean_dob(value):
    # Handle NaN
        if isinstance(value, float) and math.isnan(value):
            return None
        # Keep None as None
        if value is None:
            return None
        # Convert any valid string date to string
        return str(value)
    
    cleaned_data["DoB"] = cleaned_data["DoB"].apply(clean_dob)

    def clean_olympic(value):
    # If pandas NaN, convert to None
        if isinstance(value, float) and math.isnan(value):
            return None
        # Keep None as None
        if value is None:
            return None
        # Convert valid string to string
        return str(value)
    
    cleaned_data["Is Olympic Discipline"] = cleaned_data["Is Olympic Discipline"].apply(clean_olympic)

    # Convert Date column to ISO format
    cleaned_data['Date'] = pd.to_datetime(cleaned_data['Date'], format='%d-%b-%Y').dt.strftime('%Y-%m-%d')
    
    return cleaned_data




def print_dataframe_report(
    df: pd.DataFrame,
    n_uniques: int = 10
) -> None:
    """
    Print a human-readable report describing all columns in a DataFrame.
    """

    print("=" * 80)
    print("DATAFRAME REPORT")
    print("=" * 80)
    print(f"Rows: {len(df)}")
    print(f"Columns: {len(df.columns)}")
    print()

    for col in df.columns:
        series = df[col]

        print("-" * 80)
        print(f"Column: {col}")
        print(f"Type:   {series.dtype}")
        print()

        print(f"Non-null count : {series.notna().sum()}")
        print(f"Missing count  : {series.isna().sum()}")
        print(f"Unique values  : {series.nunique(dropna=True)}")

        uniques = series.dropna().unique().tolist()[:n_uniques]
        print(f"First {n_uniques} uniques:")
        print(indent(str(uniques), "\n"))

        if pd.api.types.is_numeric_dtype(series):
            print("\nNumeric statistics:")
            print(f"  Mean : {series.mean()}")
            print(f"  Std  : {series.std()}")
            print(f"  Min  : {series.min()}")
            print(f"  Max  : {series.max()}")
        else:
            value_counts = series.value_counts(dropna=True)

            print("\nCategorical statistics:")
            if not value_counts.empty:
                print(f"  Most frequent : {value_counts.index[0]}")
                print(f"  Frequency     : {value_counts.iloc[0]}")
            else:
                print("  Most frequent : None")

            print(f"  Min (lex)     : {series.min()}")
            print(f"  Max (lex)     : {series.max()}")

        print()

    print("=" * 80)
    print("END OF REPORT")
    print("=" * 80)


# Params normally set by user

In [90]:
# Load all data and schemas
data_schema = load_json_schema(DATA_SCHEMA_PATH)
rule_schema = load_json_schema(RULE_SCHEMA_PATH)
fulldata = load_data(DATA_PATH)
# Extract the item schema since data_schema is for an array
item_schema = data_schema.get('items', data_schema)
fulldata = clean_data(fulldata, item_schema)

# print out unique sports in the data
print("Unique sports in the data:")
print(fulldata['Sport'].unique().tolist())
print(fulldata['Rank'].unique().tolist())


Unique sports in the data:
['Biathlon', 'Bobsleigh', 'Figure Skating', 'Cross-Country Skiing', 'Ski Jumping', 'Skeleton', 'Alpine Skiing', 'Ski Mountaineering', 'Freestyle Skiing', 'Snowboard']
[11, 21, 17, 25, 26, 14, 7, 4, 5, 29, 16, 19, 20, 62, 38, 18, 27, 28, 997, 13, 59, 37, 41, 61, 6, 22, 24, 30, 3, 50, 52, 996, 10, 82, 65, 68, 90, 1, 89, 9, 42, 23, 67, 73, 47, 72, 8, 63, 70, 78, 48, 71, 15, 40, 60, 75, 44, 31, 34, 46, 58, 36, 33, 51, 55, 49, 77, 57, 111, 35, 39, 88, 81, 79, 12, 32, 43, 91, 83, 86, 69, 80, 54, 96, 2, 64, 76, 45, 53, 999, 998, 56, 66]


In [91]:
SPORT = 'Alpine Skiing'  # select a sport for testing
# get all Disciplines for the sport
# first filter data for the sport
data = fulldata[fulldata['Sport'] == SPORT]
# then get unique Disciplines
disciplines = data[data['Sport'] == SPORT]['Discipline'].unique().tolist()
print(disciplines)

DISCIPLINE = disciplines[0]  # select the first discipline for testing

# get all genders for the sport and discipline
data = data[data['Discipline'] == DISCIPLINE]
genders = data['Gender'].unique().tolist()
print(genders)

GENDER = genders[0]  # select the first gender for testing

print(f"\n\nSelected\nSPORT: {SPORT},\nDISCIPLINE:{DISCIPLINE},\nGENDER: {GENDER}")

print("\n\n")

print_dataframe_report(data)

['Downhill', 'Slalom', 'Super G', 'Giant Slalom']
['Men', 'Women']


Selected
SPORT: Alpine Skiing,
DISCIPLINE:Downhill,
GENDER: Men



DATAFRAME REPORT
Rows: 54
Columns: 11

--------------------------------------------------------------------------------
Column: Date
Type:   object

Non-null count : 54
Missing count  : 0
Unique values  : 5
First 10 uniques:

['2025-12-18', '2025-12-13', '2025-12-12', '2025-12-04', '2025-12-20']

Categorical statistics:
  Most frequent : 2025-12-20
  Frequency     : 17
  Min (lex)     : 2025-12-04
  Max (lex)     : 2025-12-20

--------------------------------------------------------------------------------
Column: Year
Type:   int64

Non-null count : 54
Missing count  : 0
Unique values  : 1
First 10 uniques:

[2025]

Numeric statistics:
  Mean : 2025.0
  Std  : 0.0
  Min  : 2025
  Max  : 2025

--------------------------------------------------------------------------------
Column: Comp.SetDetail
Type:   object

Non-null count : 54
Missing count  : 0
Un

## Validation Functions

### data validation

In [92]:
# A function that validates data against a schema
def validate_data(data, schema):
    data = data.to_dict(orient='records')
    try:
        validate(instance=data, schema=schema)
        print("Data is valid.")
        return True
    except jsonschema.exceptions.ValidationError as err:
        print("Data is invalid:", err)
        return False


In [93]:
# Extract the item schema since data_schema is for an array
vality = validate_data(data, data_schema)

Data is valid.


### rule validation

In [94]:
def normalize_rules(obj):
    """
    Recursively normalize a dict representing rules:
    1. Convert empty strings or None-like values to Python None.
    2. Convert 'created at' and 'updated at' dates to ISO strings.
    3. Convert 'version' to string.
    """
    if isinstance(obj, dict):
        new_obj = {}
        for k, v in obj.items():
            # Step 1: Normalize recursively
            v = normalize_rules(v)

            # Step 2: Handle dates
            if k in ("created at", "updated at") and isinstance(v, (date, datetime)):
                new_obj[k] = v.isoformat()
            # Step 3: Handle version field
            elif k == "version" and isinstance(v, (float, int)):
                new_obj[k] = str(v)
            # Step 4: Convert empty string or None to Python None
            elif v == "" or v is None:
                new_obj[k] = None
            else:
                new_obj[k] = v
        return new_obj
    elif isinstance(obj, list):
        return [normalize_rules(v) for v in obj]
    elif obj == "" or obj is None:
        return None
    else:
        return obj

    
def validate_rules(data: dict, schema: dict) -> bool:
    """
    Validate a Python dict against a JSON Schema.
    Prints all validation errors if they exist.
    Returns True if validation passed, False otherwise.
    """
    validator = Draft7Validator(schema)
    errors = sorted(validator.iter_errors(data), key=lambda e: e.path)

    if not errors:
        print("✅ Validation passed!")
        return True

    print("❌ Validation failed with the following errors:\n")
    for err in errors:
        path = ".".join([str(p) for p in err.path])
        print(f"- Path: {path or 'root'}")
        print(f"  Message: {err.message}\n")
    return False


In [95]:
rules = load_yaml_rules(RULE_PATH)
rules = normalize_rules(rules)
is_valid = validate_rules(rules, rule_schema)

✅ Validation passed!


## Apply Rules

In [96]:
def parse_date(d: str) -> date:
    return date.fromisoformat(d)

def filter_by_date(
    df: pd.DataFrame,
    date_rule: Union[int, List[str]]
) -> pd.DataFrame:
    """
    Filters dataframe by either a single year or a date range.
    """
    if isinstance(date_rule, int):
        return df[df["Year"] == date_rule]

    start_date, end_date = pd.to_datetime(date_rule)
    dates = pd.to_datetime(df["Date"])
    return df[(dates >= start_date) & (dates <= end_date)]

def condition_is_met(
    condition: Dict[str, Any],
    results: List[Dict[str, Any]],
) -> Tuple[int, List[Dict[str, Any]]]:
    """
    Evaluate a single condition against competition results.

    Returns:
        score (int): 0 or negative
        evidence (list): matching competition entries
    """

    start_date, end_date = map(parse_date, condition["date"])
    competitions = set(condition["competition"])
    performance_range = condition["performance"]
    min_rank = min(performance_range)
    max_rank = max(performance_range) if len(performance_range) > 1 else min_rank
    required_count = condition["count_at_least"]

    evidence = []

    for r in results:
        if r["Rank"] is None:
            continue

        r_date = parse_date(r["Date"])

        if not (start_date <= r_date <= end_date):
            continue

        if r["Comp.SetDetail"] not in competitions:
            continue

        rank = int(r["Rank"])
        if not (min_rank <= rank <= max_rank):
            continue

        evidence.append(
            {
                "competition": r["Comp.SetDetail"],
                "date": r["Date"],
                "result": rank,
            }
        )

    count = len(evidence)

    # ---- scoring logic ----
    if count >= required_count:
        score = 0
    elif count > 0:
        score = -1
    else:
        score = -required_count

    return score, evidence




def criteria_is_met(
    criteria: Dict[str, Any],
    results: List[Dict[str, Any]],
) -> Dict[str, Any]:
    """
    Evaluate a criteria by summing its condition scores.
    """

    condition_evaluations = []
    total_score = 0

    for condition in criteria["conditions"]:
        score, evidence = condition_is_met(condition, results)
        total_score += score

        condition_evaluations.append(
            {
                "condition": condition["description"],
                "result": score,
                "evidence": evidence,
            }
        )

    return {
        "criteria": criteria["description"],
        "priority": criteria["priority"],
        "result": total_score,
        "conditions": condition_evaluations,
    }


def evaluate_athlete(
    athlete: str,
    results: List[Dict[str, Any]],
    rule_set: Dict[str, Any],
) -> Dict[str, Any]:
    """
    Evaluate all criteria for a single athlete.
    """

    criteria_results = []
    passed_count = 0

    criterias = rule_set["rule_tree"]["criteria"]

    for criteria in sorted(criterias, key=lambda c: c["priority"]):
        evaluation = criteria_is_met(criteria, results)
        criteria_results.append(evaluation)

        if evaluation["result"] == 0:
            passed_count += 1

    latest_date = max(r["Date"] for r in results if r["Date"] is not None)

    return {
        "athlete": athlete,
        "date": latest_date,
        "version_criteria_set": rule_set["version"],
        "N_criteria_passed": passed_count,
        "evaluation": criteria_results,
    }


In [97]:
from typing import List, Dict, Any

def evaluate_athletes(
    athletes: List[str],
    df: pd.DataFrame,
    rule_set: Dict[str, Any],
    verbose: bool = False
) -> List[Dict[str, Any]]:
    """
    Evaluate each athlete and return a list of qualification matrices.
    Filters results to the athlete's rows before evaluation.
    """
    all_results = df.to_dict(orient='records')
    matrices: List[Dict[str, Any]] = []

    for athlete in athletes:
        if verbose:
            print(f"Evaluating: {athlete}")
        athlete_results = [r for r in all_results if r.get("Person/Team") == athlete]
        qm = evaluate_athlete(athlete=athlete, results=athlete_results, rule_set=rule_set)
        matrices.append(qm)

    return matrices

In [99]:
import pprint
athletes = data['Person/Team'].unique().tolist()
results = evaluate_athletes(athletes, data, rules, False)
pprint.pprint(results)

[{'N_criteria_passed': 2,
  'athlete': 'Marco Odermatt (SUI, 08 Oct 1997)',
  'date': '2025-12-20',
  'evaluation': [{'conditions': [{'condition': 'Top-7 WC 2025/26',
                                  'evidence': [{'competition': 'Audi FIS Ski '
                                                               'World Cup',
                                                'date': '2025-12-18',
                                                'result': 1},
                                               {'competition': 'Audi FIS Ski '
                                                               'World Cup',
                                                'date': '2025-12-04',
                                                'result': 1},
                                               {'competition': 'Audi FIS Ski '
                                                               'World Cup',
                                                'date': '2025-12-20',
                                  